# Inspect Gold Star Schema

Notebook para validar a Gold dimensional da V2.

Por padrao, tenta abrir primeiro a amostra `dev`. Se ela nao existir, tenta abrir a Gold oficial.

In [ ]:
from v2.config.paths import DELTA_ROOT, star_schema_gold_dir

official_root = star_schema_gold_dir(2025)
dev_root = DELTA_ROOT / "dev" / "gold" / "star_schema" / "2025_01"

root = dev_root if (dev_root / "fact_trips" / "_delta_log").exists() else official_root
table_names = ["dim_data", "dim_clima", "dim_localizacao", "fact_trips"]

print(f"Official root: {official_root}")
print(f"Dev root     : {dev_root}")
print(f"Using root   : {root}")

missing_tables = [
    table_name
    for table_name in table_names
    if not (root / table_name / "_delta_log").exists()
]

if missing_tables:
    raise FileNotFoundError(
        f"Tabelas Gold nao encontradas em {root}: {missing_tables}"
    )

In [ ]:
from v2.config.spark import create_spark

spark = create_spark("NotebookInspectGoldStarSchema")

In [ ]:
dfs = {
    table_name: spark.read.format("delta").load(str(root / table_name))
    for table_name in table_names
}

dim_data = dfs["dim_data"]
dim_clima = dfs["dim_clima"]
dim_localizacao = dfs["dim_localizacao"]
fact_trips = dfs["fact_trips"]

In [ ]:
for table_name, df in dfs.items():
    print(f"\n{table_name}")
    df.printSchema()

In [ ]:
summary = spark.createDataFrame(
    [(table_name, df.count()) for table_name, df in dfs.items()],
    ["tabela", "linhas"],
)

summary.show(truncate=False)

## Cobertura de datas

In [ ]:
from pyspark.sql import functions as F

dim_data.select(
    F.count("*").alias("linhas"),
    F.expr("count(distinct data)").alias("dias_distintos"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
).show(truncate=False)

dim_clima.select(
    F.count("*").alias("linhas"),
    F.expr("count(distinct data)").alias("dias_distintos"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
    F.sum(F.col("registro_clima_incompleto").cast("int")).alias(
        "dias_clima_incompleto"
    ),
).show(truncate=False)

## Chaves nulas na fato

In [ ]:
fact_trips.select(
    F.count("*").alias("linhas"),
    F.sum(F.when(F.col("data_id").isNull(), 1).otherwise(0)).alias("data_id_nulo"),
    F.sum(F.when(F.col("clima_id").isNull(), 1).otherwise(0)).alias("clima_id_nulo"),
    F.sum(
        F.when(F.col("localizacao_partida_id").isNull(), 1).otherwise(0)
    ).alias("localizacao_partida_id_nulo"),
    F.sum(
        F.when(F.col("localizacao_chegada_id").isNull(), 1).otherwise(0)
    ).alias("localizacao_chegada_id_nulo"),
).show(truncate=False)

## Integridade dos relacionamentos

In [ ]:
orfaos_data = fact_trips.select("data_id").distinct().join(
    dim_data.select("data_id").distinct(), on="data_id", how="left_anti"
).count()

orfaos_clima = fact_trips.select("clima_id").distinct().join(
    dim_clima.select("clima_id").distinct(), on="clima_id", how="left_anti"
).count()

orfaos_partida = fact_trips.select(
    F.col("localizacao_partida_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

orfaos_chegada = fact_trips.select(
    F.col("localizacao_chegada_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

spark.createDataFrame(
    [
        ("fact_trips -> dim_data", orfaos_data),
        ("fact_trips -> dim_clima", orfaos_clima),
        ("fact_trips -> dim_localizacao partida", orfaos_partida),
        ("fact_trips -> dim_localizacao chegada", orfaos_chegada),
    ],
    ["relacionamento", "chaves_orfas"],
).show(truncate=False)

## Amostra enriquecida

In [ ]:
fact_trips.alias("f").join(
    dim_data.alias("d"), on="data_id", how="left"
).join(
    dim_clima.alias("c"), on="clima_id", how="left"
).select(
    F.col("f.data_hora_partida"),
    F.col("d.data").alias("data_viagem"),
    F.col("f.duracao_minutos"),
    F.col("f.qtd_passageiros"),
    F.col("f.tipo_pagamento_desc"),
    F.col("f.distancia_km"),
    F.col("f.valor_total"),
    F.col("c.temp_media_c"),
    F.col("c.precipitacao_mm"),
    F.col("c.categoria_chuva"),
    F.col("c.registro_clima_incompleto"),
).orderBy("data_hora_partida").show(20, truncate=False)

In [ ]:
spark.stop()